# Drinking Water Demand Forecasting with Recurrent Neural Networks - Part 2
## Multi-step ahead prediction with Recurrent Neural Networks

In Part 2, we will explore **multi-step ahead prediction** using Recurrent Neural Networks (RNN) with PyTorch. 

We will focus on forecasting future values of the water demand time series in the next 24 hours. We will use two methods: 

1. Using the models trained in <u>Part 1</u> for one-step ahead in a *recursive* (or autoregressive) fashion; this method is explained in the walkthrough.
2. Using models specifically trained to *predict 24-steps ahead in one go*; you will be performing this step as part of the assignments. 

Please go through the Walkthrough sections of this notebooks, until you reach the Assignments.

## Load python modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
from urllib.request import urlretrieve

from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Check if CUDA is available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Download and load the data

In [ ]:
# Download the water demand dataset
year_1 = "https://surfdrive.surf.nl/files/index.php/s/T2SN8dnXXYWz4q5/download"
year_2 = "https://surfdrive.surf.nl/files/index.php/s/StRwCboQoDPRGdD/download"

data_folder = "data"
water_demand_file1 = os.path.join(data_folder, "water_demand_Y1.txt")
water_demand_file2 = os.path.join(data_folder, "water_demand_Y2.txt")

if not os.path.isfile(water_demand_file1):
    print("Downloading dataset...")
    os.makedirs("data", exist_ok=True)
    urlretrieve(year_1, water_demand_file1)
    urlretrieve(year_2, water_demand_file2)

In [ ]:
# load water demand data
df_year1 = pd.read_csv(water_demand_file1,header=None, sep=r"\s+")
df_year2 = pd.read_csv(water_demand_file2,header=None, sep=r"\s+")
df = pd.concat([df_year1, df_year2], axis = 0).reset_index(drop=True)
df.head()

## Creation of dataset and data normalization/standardization

In [ ]:
def create_sequences(series,T=168,H=24):
    # This function creates a dataset of input/output sequences from a time series.
    # The input sequence is T steps long, from time t to time t+T (excluded).
    # The output sequence is H steps long, from time t+T to time t+T+H (excluded).
    X = []
    Y = []
    for t in range(len(series)-T-H):
        x = series[t:t+T]
        X.append(x)
        y = series[t+T:t+T+H]
        Y.append(y)
    X = np.array(X)
    Y = np.array(Y)
    return X,Y

def scale_sequences(X,scaler=None,scaler_type='standard'):
    # Uses a standard scaler to transform sequences. The scaler is created if no scaler is passed as argument.
    Xshape=X.shape
    if scaler:
        X = scaler.transform(X.reshape(-1,1)).reshape(Xshape)
        return X
    else:
        if scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'minmax':
            scaler = MinMaxScaler()
        else:
            raise Exception("Type of scikit-learn scaler not supported. Choose 'standard' or 'minmax.")
        X = scaler.fit_transform(X.reshape(-1,1)).reshape(Xshape)
        return X, scaler

We increase the predictive horizon from 1 to 24 steps ahead

In [ ]:
T = 168     # number of time steps to use for prediction (168 hours = 1 week)
H = 24      # number of time steps to predict (1 day)
X, Y = create_sequences(df.values.reshape(-1),T=T,H=H)
print(X.shape)
print(Y.shape)

Now each input-ouput pair is made of: 1) an input sequence of 168 hours (e.g., 1 week) and 2) an output sequence of 24 hours.

In [ ]:
random_ix = np.random.choice(X.shape[0])
f, ax = plt.subplots(1,figsize=(10,3))
ax.plot(np.arange(T),X[0], label='X (input)')
ax.plot(np.arange(T,T+H),Y[0], 'x', label = 'Y (output)')
ax.set_title(f'X and Y for sequence #{random_ix} of the Year #1 dataset');
ax.legend();

These sequences are not normalized yet, and we need to split them into training and validation. We first perform the latter task and then we use ```scale_sequences``` to perform the scaling. Is important to use the scaler "fitted" for the training dataset to scale tha validation dataset and then the test dataset). Since the input and output variables are both water demand, we only need to fit one scaler. ```scale_sequences``` recognize whether to fit a scaler or use an existing one based on the arguments it receives when called.

In [ ]:
# We keep track of indexes of train and validation.
X_tra, X_tst, Y_tra, Y_tst, ix_tra, ix_tst = train_test_split(
    X, Y, np.arange(X.shape[0]), test_size=0.30, shuffle=True, random_state=42)

# Split the existing test dataset into validation and test sets (50/50 split)
X_val, X_tst, Y_val, Y_tst, ix_val, ix_tst = train_test_split(
    X_tst, Y_tst, ix_tst, test_size=0.5, shuffle=True, random_state=42)


print(f"X_tra.shape: {X_tra.shape}, Y_tra.shape: {Y_tra.shape}")
print(f"X_val.shape: {X_val.shape}, Y_val.shape: {Y_val.shape}")
print(f"X_tst.shape: {X_tst.shape}, Y_tst.shape: {Y_tst.shape}")

Now we scale all data. Remember: <u>you "fit" the scaler using training data </u>; when you develop your machine learning models, you have to assume that you have no information on validation and test data.

In [ ]:
# scale/normalize
X_tra, scaler = scale_sequences(X_tra, scaler_type='standard')
Y_tra = scale_sequences(Y_tra, scaler)
X_val = scale_sequences(X_val, scaler)
Y_val = scale_sequences(Y_val, scaler)
X_tst = scale_sequences(X_tst, scaler)
Y_tst = scale_sequences(Y_tst, scaler)

## Definition of MultiLayer Perceptron and Simple Recurrent Neural Network

To load the models trained in Part 1, <u>we need the same classes</u> implemented there. Here we do it for MLP and SimpleRNN. You can do the same for other classes, if you implemented them.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(MLP, self).__init__()
        # Define the layers of the network
        self.fc1 = nn.Linear(T, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Define the forward pass
        x = F.tanh(self.fc1(x))  # Activation function (ReLU) after first layer
        x = F.tanh(self.fc2(x))  # Activation function (ReLU) after second layer
        x = self.fc3(x)          # Output layer
        return x

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        # RNN layer
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)

        # Output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Reshape input to have feature dimension of 1
        x = x.unsqueeze(-1)   # Assuming input x has shape (batch, sequence)

        # RNN layer
        x, hn = self.rnn(x)   # We do not need the hidden states hn

        # Select the output of the last time step
        x = x[:, -1, :]

        # Output layer
        x = self.fc(x)

        return x

### Datasets and Data Loaders


We create the `TensorDatasets` and the `DataLoaders`.

In [ ]:
train_dataset = TensorDataset(torch.tensor(X_tra, dtype=torch.float32), torch.tensor(Y_tra, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_tst, dtype=torch.float32), torch.tensor(Y_tst, dtype=torch.float32 ))

In [ ]:
batch_size = 256      # You can modify this based on your requirements

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Recursive multi-step ahead prediction

Here, we will evaluate how the models built for one-step ahead prediction perform when used recursively to forecast over longer horizons.

### Function for recursive multi-step ahead prediction 

The code below defines the `evaluate_model_multistep` function that performs recursive multi-step ahead prediction and evaluates model's performance.

Below is a step by step description of each line of code, divided in parts. 

<u>Function Definition</u>
- `model`: The neural network model to be evaluated.
- `test_loader`: A DataLoader containing the test dataset.
- `criterion`: The loss function for evaluating performance.
- `device`: The computing device (CPU/GPU) for model evaluation.
- `T`: The length of the input sequence.
- `H`: The number of prediction steps ahead.

<u>Model Evaluation Setup</u>
- `model.eval()`: Sets the model to evaluation mode.
- `test_loss`: Accumulates the total loss over the test dataset.

<u>Storing Predictions and Targets</u>
- `all_predictions` and `all_targets`: Lists to store predictions and actual targets from the test dataset.

<u>Evaluation Loop</u>
- `with torch.no_grad()`: Disables gradient calculations since they are not needed.
- Iterates over the test dataset to fetch batches of inputs and targets.

<u>Multi-Step Prediction Process</u>
- Iterates `H` times for each batch to simulate multi-step prediction.
- `step_inputs`: Current input for the model.
- `targets`: Actual target values for prediction.
- Updates `step_inputs` with the model's output for each step.

<u>Loss Calculation and Storage</u>
- After `H` steps, concatenates predictions to compare against actual targets.
- Adds batch loss to `test_loss`.
- Stores predictions and targets for each batch in respective lists.
  
<u>Final Output</u>
- Calculates and returns the average test loss.
- Returns tensors of all predictions and all targets from the test dataset.all targets from the test dataset.all targets from the test dataset.ased on past observations is crucial.


In [ ]:
def evaluate_model_multistep(model, test_loader, criterion, device, T, H):
    model.eval()  # Set the model to evaluation mode
    test_loss = 0

    all_predictions = []  # List to store all batch predictions
    all_targets = []      # List to store all batch targets

    with torch.no_grad():  # No need to track gradients during evaluation
        for initial_inputs, initial_targets in test_loader:
            step_inputs = initial_inputs.to(device)
            targets = initial_targets.to(device)

            # Holds predictions for comparison with targets
            predictions = []

            # Iterate for H steps
            for h in range(H):
                outputs = model(step_inputs)
                predictions.append(outputs)

                # Reshape or expand outputs to be 3D: [batch_size, 1, features]
                next_input = outputs.unsqueeze(-1).squeeze(1)  # Adjust the dimensions as necessary

                # Update step_inputs by sliding the window: remove the oldest input and add the new output
                # Ensure that step_inputs and next_input are correctly shaped for concatenation
                step_inputs = torch.cat((step_inputs[:, 1:], next_input), dim=1)

            # Concatenate predictions and calculate loss against the entire target sequence
            batch_predictions  = torch.stack(predictions, dim=1).squeeze(-1)  # Squeeze the last dimension

            loss = criterion(batch_predictions , targets)
            test_loss += loss.item()

            # Store batch predictions and targets
            all_predictions.append(batch_predictions.cpu())
            all_targets.append(targets.cpu())

    # Concatenate all batch predictions and targets into tensors
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    avg_test_loss = test_loss / len(test_loader)
    return avg_test_loss, all_predictions, all_targets

### Load the MLP and RNN models for one-step ahead prediction

Make sure to point at the <u>correct folder path</u> in the hard drive to load the models. Also, make sure you use the <u>same number of neurons</u> when instantiating the architecture.

In [ ]:
model_MLP = MLP(256,1).to(device)
MLP_load_path = './models/MLP_model.pth'
model_MLP.load_state_dict(torch.load(MLP_load_path, map_location=torch.device(device)))

In [ ]:
model_RNN = SimpleRNN(128,1).to(device)
RNN_load_path = './models/RNN_model.pth'
model_RNN.load_state_dict(torch.load(RNN_load_path, map_location=torch.device(device)))

### Predict over the extended horizon on the test dataset

In [ ]:
criterion = nn.MSELoss()  # Or another appropriate loss function

avg_test_loss_MLP, predictions_MLP, _ = evaluate_model_multistep(model_MLP, test_loader, criterion, device, T, H)
avg_test_loss_RNN, predictions_RNN, targets = evaluate_model_multistep(model_RNN, test_loader, criterion, device, T, H)

### Comparison of performance 

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"MLP --> num. trainable parameters:{count_parameters(model_MLP):8d} | Test loss: {avg_test_loss_MLP:.4f}")
print(f"RNN --> num. trainable parameters:{count_parameters(model_RNN):8d} | Test loss: {avg_test_loss_RNN:.4f}")

In [ ]:
# Create subplots
f, axes = plt.subplots(2, 4, figsize=(20, 8))

for ix,ax in enumerate(axes.reshape(-1)):
    random_index = random.randint(0, len(targets) - 1)
    # Rescaling targets and predictions
    target_rescaled = scaler.inverse_transform(targets[random_index].cpu().numpy().flatten().reshape(-1, 1)).flatten()
    prediction_MLP_rescaled = scaler.inverse_transform(predictions_MLP[random_index].cpu().numpy().flatten().reshape(-1, 1)).flatten()
    prediction_RNN_rescaled = scaler.inverse_transform(predictions_RNN[random_index].cpu().numpy().flatten().reshape(-1, 1)).flatten()

    # Plotting prediction vs target
    ax.scatter(np.arange(T, T+H), target_rescaled, marker='x', label='Actual')
    ax.scatter(np.arange(T, T+H), prediction_MLP_rescaled, marker='o', facecolors='none', edgecolors='r', label='Predicted MLP')
    ax.scatter(np.arange(T, T+H), prediction_RNN_rescaled, marker='o', facecolors='none', edgecolors='g', label='Predicted RNN')

    ax.set_title(f'Test Example #{random_index}')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Value')
    if ix == 0:
        ax.legend()

f.tight_layout()

## Assignments

1. **Comparison between RNN and MLP for recursive prediction**
  - Critically analyze the performance differences between the RNN and MLP architecture for the recursive prediction. Which model yielded better results? How does this compare with the results you obtained for Part 1, i.e., one-step ahead prediction? Discuss the reasons for this performance difference.

2. **Develop MLP and RNN models for 24-steps ahead forecasting**
  - Extend the MLP and RNN models that you developed in Part 1 for 24-steps ahead prediction. 
  - Minimal changes to the existing implementations are required:
    - Make sure `output_size=24`; 
    - The training loop function used in Part 1 can be reused for this task;
    - Similarly, you have to use the same function of Part 1 for model evaluation;
    - `evaluate_model_multistep` will not work!
  - Discuss the results, also by comparing them against the recursive approach in the walkthorugh.
  
3. **[OPTIONAL] Deeper RNNs**
  - Adapt the simple RNN model to an RNN with two recurrent layers; check if this captures better the temporal dependencies in the data.
    - Modify the `SimpleRNN` class to a `TwoLayerRNN` by setting `num_layers=2` in the RNN layer (`nn.RNN`). This change enables the network to have two RNN layers, potentially enhancing its capacity to capture complex patterns in the data. See [here](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) for further info.
  -  After trying a few alternative configurations (e.g., more or less neurons) and tweaking some training hyperparameters (e.g., learning rate, number of epochs), discuss whether the deeper architecture better captures the temporal relationships.

 
4. **[OPTIONAL] Develop an Encoder-Decoder RNN model for 24-steps ahead forecasting**
  - Adapt the simple RNN model to an RNN encoder-decoder framework, focusing on managing hidden states for producing the 24-steps ahead forecasts.
    - Encoder Modifications: Retain the basic structure of the `SimpleRNN` class. This time, the hidden states `hn` from the RNN layer are crucial as they capture the *context*.
    - Decoder Design: In the decoder, use the context in the final hidden state from the encoder as the initial state. Generate the forecast sequence step by step.
  - After trying a few version of this architecture and tweaking some hyperparameters, discuss whether the encoder-decoder better captures the temporal relationships.